# Notebook 1: Your First Agent

**What you'll learn:**
- What an agent is and why it's different from a plain model call
- How to create an agent and call it
- How to inspect the result (AgentResult)
- How conversation memory works (agent.messages)
- What attributes the Agent class holds

**Prerequisites:**
- Read `00-python-concepts-for-sdk.md` (async, ABC, decorators, composition)
- AWS credentials configured (for Amazon Bedrock)
- `pip install strands-agents`

**Companion reading:** `01-what-is-an-agent.md`

---
## What is an Agent?

A **plain model call** sends text to an AI and gets text back. That's it. Like texting a friend.

An **agent** is much more -- it's an AI that can **think**, **act**, and **remember**:

| Part | Analogy | What it does |
|------|---------|-------------|
| **Model** | Brain | Thinks, understands language, generates responses |
| **Tools** | Hands | Performs actions (check weather, query database, send email) |
| **Messages** | Memory | Remembers everything said in the conversation |
| **Hooks** | Reflexes | Automatic reactions at key moments (log events, guard against bad actions) |
| **System prompt** | Personality | Permanent instructions ("You are a helpful assistant") |
| **Conversation manager** | Forgetting old memories | Prevents memory from overflowing |

The key difference: **the agent decides what to do**. It reads tool descriptions, picks the right one, generates arguments, reads results, and formulates a response. The SDK orchestrates this entire process.

In [ ]:
# ============================================================
# STEP 1: Create the simplest possible agent
# ============================================================

# Import the Agent class -- this is the main class in the SDK.
# It lives in src/strands/agent/agent.py
from strands import Agent

# Create an agent with all defaults.
# Behind the scenes, this sets up:
#   - model: BedrockModel() (Amazon Bedrock with Claude)
#   - system_prompt: a default helpful assistant prompt
#   - tools: none (no tools registered)
#   - messages: [] (empty conversation history)
#   - callback_handler: built-in handler that prints to console
#   - conversation_manager: SlidingWindowConversationManager (manages memory size)
agent = Agent()

# Let's see what model is being used.
# get_config() returns a dictionary of model settings.
print("Model config:")
print(agent.model.get_config())

In [ ]:
# ============================================================
# STEP 2: Call the agent with a simple question
# ============================================================

# When you write agent("..."), Python calls the __call__ method.
# This triggers an 8-function chain (see 01-what-is-an-agent.md).
#
# What happens internally:
#   1. __call__          -> converts to async
#   2. invoke_async      -> collects all streaming events
#   3. stream_async      -> acquires lock, converts prompt to message
#   4. _run_loop         -> fires hooks, appends message, starts event loop
#   5. event_loop_cycle  -> calls the AI model
#   6. Model responds with stop_reason="end_turn" (no tools needed)
#   7. EventLoopStopEvent is yielded
#   8. AgentResult is created and returned
#
# You'll see text streaming in real-time below (word by word).
# That's the callback_handler printing each StreamEvent.

result = agent("What is 2+2? Answer in one sentence.")

---
## What is AgentResult?

When the agent finishes, it returns an `AgentResult` object. This object contains everything about what happened:

| Field | Type | What it means |
|-------|------|--------------|
| `stop_reason` | `str` | Why the agent stopped: `"end_turn"` (finished normally) or `"interrupt"` (paused for human input) |
| `message` | `dict` | The model's final message (contains the response text and any tool calls) |
| `metrics` | `EventLoopMetrics` | Performance stats: how long it took, how many tokens were used |
| `state` | `dict` | Internal event loop state |

**Source:** `src/strands/agent/agent_result.py`

The `AgentResult` also has a `__str__` method that extracts just the text, so `print(result)` gives you the response text.

In [ ]:
# ============================================================
# STEP 3: Inspect the AgentResult
# ============================================================

# --- stop_reason ---
# "end_turn" means the model finished speaking normally.
# Other possible values: "tool_use" (wants to call tools), "interrupt" (paused)
print("=== stop_reason ===")
print(result.stop_reason)
print()

# --- message ---
# The model's last message. It's a dictionary with:
#   "role": "assistant"  (the model is the assistant)
#   "content": [...]     (list of content blocks)
print("=== message ===")
print(result.message)
print()

# --- metrics ---
# Performance data about the event loop run.
print("=== metrics ===")
print(f"Input tokens:  {result.metrics.accumulated_usage.get('inputTokens', 'N/A')}")
print(f"Output tokens: {result.metrics.accumulated_usage.get('outputTokens', 'N/A')}")
print(f"Latency (ms):  {result.metrics.accumulated_metrics.get('latencyMs', 'N/A')}")
print()

# --- str(result) ---
# The __str__ method extracts just the text from the message.
# This is the easiest way to get the response.
print("=== str(result) ===")
print(str(result))

---
## What is agent.messages? (Conversation Memory)

Every time you call the agent, messages are added to `agent.messages`. This is the agent's **memory** of the conversation.

After the call above, `agent.messages` should contain **2 messages**:
1. **User message** -- your question ("What is 2+2?")
2. **Assistant message** -- the model's response

Each message is a dictionary with:
- `"role"`: who said it (`"user"` or `"assistant"`)
- `"content"`: list of content blocks (text, images, tool calls, etc.)

The model sees ALL previous messages every time it's called. That's how it "remembers" the conversation.

In [ ]:
# ============================================================
# STEP 4: Look at the conversation memory
# ============================================================

import json  # json.dumps makes dictionaries look nicer when printed

# agent.messages is a Python list of dictionaries.
# After one call, there should be 2 messages.
print(f"Number of messages: {len(agent.messages)}")
print()

# Walk through each message
for i, msg in enumerate(agent.messages):
    print(f"--- Message {i} ---")
    print(f"Role: {msg['role']}")  # "user" or "assistant"
    
    # The content is a list of content blocks.
    # For a simple text exchange, each block has a "text" key.
    for block in msg['content']:
        if 'text' in block:
            # Truncate long text for readability
            text = block['text']
            preview = text[:100] + "..." if len(text) > 100 else text
            print(f"Text: {preview}")
    print()

In [ ]:
# ============================================================
# STEP 5: Call the agent again -- watch the history grow
# ============================================================

# This is a follow-up question. The agent will "remember" the first exchange
# because all previous messages are sent to the model.
result2 = agent("What about 3+3? Answer in one sentence.")

In [ ]:
# ============================================================
# STEP 6: Check the conversation history after 2 calls
# ============================================================

# After 2 calls, there should be 4 messages:
#   [0] user:      "What is 2+2?"
#   [1] assistant:  "2+2 is 4"
#   [2] user:      "What about 3+3?"
#   [3] assistant:  "3+3 is 6"

print(f"Number of messages after 2 calls: {len(agent.messages)}")
print()

for i, msg in enumerate(agent.messages):
    # Extract the text from the first content block
    text = ""
    for block in msg['content']:
        if 'text' in block:
            text = block['text'][:80]  # First 80 characters
            break
    print(f"[{i}] {msg['role']:10s} | {text}")

---
## Agent Attributes -- What the Agent Holds

When you create `Agent()`, many attributes are set up. Here are the key ones:

| Attribute | What it is | Analogy |
|-----------|-----------|--------|
| `agent.model` | The AI model being used | The brain |
| `agent.system_prompt` | Permanent instructions | The personality |
| `agent.messages` | Conversation history | The memory |
| `agent.tool_registry` | All registered tools | The catalog of buttons |
| `agent.callback_handler` | Handles streaming output | The live transcriber |
| `agent.conversation_manager` | Manages memory size | The librarian |
| `agent.state` | Custom data storage | The scratch pad |
| `agent.hooks` | Lifecycle callbacks | The sensors |
| `agent.event_loop_metrics` | Performance stats | The stopwatch |

**Source:** `src/strands/agent/agent.py` (lines 86-250)

In [ ]:
# ============================================================
# STEP 7: Inspect key agent attributes
# ============================================================

# --- model ---
# This is the AI model adapter (translator between SDK and provider).
# Default is BedrockModel, which connects to Amazon Bedrock.
print("=== Model ===")
print(f"Type: {type(agent.model).__name__}")  # BedrockModel
print(f"Config: {agent.model.get_config()}")
print()

# --- system_prompt ---
# Permanent instructions sent with every model call.
# This is NOT stored in agent.messages -- it's sent separately.
print("=== System Prompt ===")
# Show first 200 characters of the system prompt
print(agent.system_prompt[:200] if agent.system_prompt else "(none)")
print()

# --- tool_registry ---
# A dictionary of registered tools. We haven't added any tools yet,
# so it should be empty.
print("=== Tool Registry ===")
print(f"Registered tools: {list(agent.tool_registry.registry.keys())}")
print()

# --- messages ---
# The conversation history (we already inspected this above).
print("=== Messages ===")
print(f"Message count: {len(agent.messages)}")
print()

# --- callback_handler ---
# The function that handles streaming output.
# Default prints text to the console as it arrives.
print("=== Callback Handler ===")
print(f"Type: {type(agent.callback_handler).__name__}")
print()

# --- conversation_manager ---
# Manages message history size to prevent overflow.
print("=== Conversation Manager ===")
print(f"Type: {type(agent.conversation_manager).__name__}")
print()

# --- state ---
# A dictionary you can use to store custom data.
# It's empty by default.
print("=== State ===")
print(f"State: {agent.state}")

---
## The Function Call Chain

When you write `result = agent("hello")`, here's what happens internally -- **8 functions** are called in sequence:

```
agent("hello")
  |
  v
__call__                    # Entry point (converts sync to async)
  |
  v
invoke_async                # Collects all events into AgentResult
  |
  v
stream_async                # Acquires lock, converts prompt to message
  |
  v
_run_loop                   # Fires before/after hooks, starts event loop
  |
  v
_execute_event_loop_cycle   # Bridge to the event loop module
  |
  v
event_loop_cycle            # THE CORE -- calls model, handles response
  |
  +-- Model responds with "end_turn"
  |   (no tools needed)
  |
  v
EventLoopStopEvent          # Signals we're done
  |
  v
AgentResult                 # Returned to you
```

In this notebook, the flow was simple because we had no tools. In Notebook 2, you'll see what happens when the model wants to use tools -- the event loop **recurses** (calls itself again).

**Source locations:**
- `__call__`: `agent.py:335`
- `invoke_async`: `agent.py:376`
- `stream_async`: `agent.py:539`
- `_run_loop`: `agent.py:643`
- `event_loop_cycle`: `event_loop.py:78`

---
## Summary

What you learned in this notebook:

- **Agent** = model (brain) + tools (hands) + messages (memory) + hooks (reflexes)
- **Creating an agent:** `agent = Agent()` sets up all defaults
- **Calling an agent:** `result = agent("question")` triggers an 8-function call chain
- **AgentResult** has: `stop_reason`, `message`, `metrics`, and text via `str(result)`
- **agent.messages** is a list that grows with each call -- the agent's memory
- **Key attributes:** model, system_prompt, messages, tool_registry, callback_handler, conversation_manager, state

**Next:** [NB2_The_Event_Loop.ipynb](./NB2_The_Event_Loop.ipynb) -- See the event loop in action with tools and logging